In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/simulated_theft_data.csv")



In [2]:


df.head()

,x_Timestamp,meter,t_kWh,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),hour,day,month,day_of_week,theft
0,2021-01-02 00:00:00,BR02,0.00850,253.9020,0.7160,50.0535,0,2,1,5,0
1,2021-01-02 01:00:00,BR02,0.00105,255.0410,0.1425,50.0170,1,2,1,5,0
2,2021-01-02 02:00:00,BR02,0.00100,255.4345,0.1335,49.9955,2,2,1,5,0
3,2021-01-02 03:00:00,BR02,0.00100,255.7510,0.1320,50.0160,3,2,1,5,0
4,2021-01-02 04:00:00,BR02,0.00100,255.8580,0.1300,50.0285,4,2,1,5,0


In [3]:
df = df.rename(columns={
    'x_Timestamp': 'timestamp',
    'meter': 'meter_id',
    't_kWh': 'consumption'
})

df['timestamp'] = pd.to_datetime(df['timestamp'])

df = df.sort_values(['meter_id', 'timestamp']).reset_index(drop=True)

df.head()

,timestamp,meter_id,consumption,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),hour,day,month,day_of_week,theft
0,2021-01-02 00:00:00,BR02,0.00850,253.9020,0.7160,50.0535,0,2,1,5,0
1,2021-01-02 01:00:00,BR02,0.00105,255.0410,0.1425,50.0170,1,2,1,5,0
2,2021-01-02 02:00:00,BR02,0.00100,255.4345,0.1335,49.9955,2,2,1,5,0
3,2021-01-02 03:00:00,BR02,0.00100,255.7510,0.1320,50.0160,3,2,1,5,0
4,2021-01-02 04:00:00,BR02,0.00100,255.8580,0.1300,50.0285,4,2,1,5,0


In [4]:
df.isnull().sum()

timestamp               0
meter_id                0
consumption             0
z_Avg Voltage (Volt)    0
z_Avg Current (Amp)     0
y_Freq (Hz)             0
hour                    0
day                     0
month                   0
day_of_week             0
theft                   0
dtype: int64

In [5]:
# Check number of rows per meter
meter_counts = df.groupby('meter_id').size()

print(meter_counts.describe())
meter_counts.head()

count      29.000000
mean     5704.689655
std      2301.476003
min       954.000000
25%      4856.000000
50%      7054.000000
75%      7208.000000
max      7274.000000
dtype: float64


meter_id
BR02     954
BR04    6296
BR05    2538
BR06    7208
BR08    1062
dtype: int64

In [6]:
# Check time gaps for ONE meter (sample)
sample_meter = df['meter_id'].iloc[0]

sample_df = df[df['meter_id'] == sample_meter].copy()
sample_df = sample_df.sort_values('timestamp')

# time difference
sample_df['diff'] = sample_df['timestamp'].diff()

sample_df['diff'].value_counts().head()

diff
0 days 01:00:00    944
0 days 23:00:00      5
9 days 23:00:00      1
2 days 23:00:00      1
1 days 23:00:00      1
Name: count, dtype: int64

In [8]:
fixed_data = []

for meter_id, group in df.groupby('meter_id'):
    
    group = group.sort_values('timestamp').set_index('timestamp')
    
    # Create full hourly index
    full_index = pd.date_range(start=group.index.min(),
                               end=group.index.max(),
                               freq='H')
    
    group = group.reindex(full_index)
    
    # Fill meter_id
    group['meter_id'] = meter_id
    
    # Fill missing values
    group['consumption'] = group['consumption'].fillna(0)
    group['theft'] = group['theft'].fillna(0)
    
    # Forward fill optional columns
    group['z_Avg Voltage (Volt)'] = group['z_Avg Voltage (Volt)'].fillna(method='ffill')
    group['z_Avg Current (Amp)'] = group['z_Avg Current (Amp)'].fillna(method='ffill')
    group['y_Freq (Hz)'] = group['y_Freq (Hz)'].fillna(method='ffill')
    
    group = group.reset_index().rename(columns={'index': 'timestamp'})
    
    fixed_data.append(group)

df_fixed = pd.concat(fixed_data).reset_index(drop=True)

In [9]:
df_fixed['hour'] = df_fixed['timestamp'].dt.hour
df_fixed['day'] = df_fixed['timestamp'].dt.day
df_fixed['month'] = df_fixed['timestamp'].dt.month
df_fixed['day_of_week'] = df_fixed['timestamp'].dt.dayofweek

In [10]:
sample_meter = df_fixed['meter_id'].iloc[0]

sample_df = df_fixed[df_fixed['meter_id'] == sample_meter].copy()
sample_df = sample_df.sort_values('timestamp')

sample_df['diff'] = sample_df['timestamp'].diff()

sample_df['diff'].value_counts().head()

diff
0 days 01:00:00    1559
Name: count, dtype: int64

In [11]:
WINDOW_SIZE = 24

# Pick ONE meter for testing
sample_meter = df_fixed['meter_id'].iloc[0]

sample_df = df_fixed[df_fixed['meter_id'] == sample_meter].copy()
sample_df = sample_df.sort_values('timestamp').reset_index(drop=True)

windows = []

for i in range(0, len(sample_df) - WINDOW_SIZE + 1, WINDOW_SIZE):
    window = sample_df.iloc[i:i + WINDOW_SIZE]
    
    if len(window) == WINDOW_SIZE:
        windows.append(window)

print("Total windows:", len(windows))

Total windows: 65


In [15]:
windows[0] #.head()

,timestamp,meter_id,consumption,z_Avg Voltage (Volt),z_Avg Current (Amp),y_Freq (Hz),hour,day,month,day_of_week,theft
0,2021-01-02 00:00:00,BR02,0.00850,253.9020,0.7160,50.0535,0,2,1,5,0.0
1,2021-01-02 01:00:00,BR02,0.00105,255.0410,0.1425,50.0170,1,2,1,5,0.0
2,2021-01-02 02:00:00,BR02,0.00100,255.4345,0.1335,49.9955,2,2,1,5,0.0
3,2021-01-02 03:00:00,BR02,0.00100,255.7510,0.1320,50.0160,3,2,1,5,0.0
4,2021-01-02 04:00:00,BR02,0.00100,255.8580,0.1300,50.0285,4,2,1,5,0.0
5,2021-01-02 05:00:00,BR02,0.00100,255.1090,0.1310,50.0250,5,2,1,5,0.0
6,2021-01-02 06:00:00,BR02,0.01055,254.4515,0.9300,50.0760,6,2,1,5,0.0
7,2021-01-02 07:00:00,BR02,0.02760,251.9020,2.2095,50.0415,7,2,1,5,0.0
8,2021-01-02 08:00:00,BR02,0.00580,248.4165,0.5910,50.0025,8,2,1,5,0.0
9,2021-01-02 09:00:00,BR02,0.00280,240.5160,0.2845,49.9950,9,2,1,5,0.0


In [13]:
windows[0]['timestamp'].diff().value_counts()

timestamp
0 days 01:00:00    23
Name: count, dtype: int64

In [ ]:
#Feature Extraction

In [ ]:
#Feature Function
def extract_features(window):
    features = {}

    consumption = window['consumption'].values

    # --- Statistical ---
    features['mean'] = np.mean(consumption)
    features['std'] = np.std(consumption)
    features['min'] = np.min(consumption)
    features['max'] = np.max(consumption)
    features['median'] = np.median(consumption)
    features['range'] = features['max'] - features['min']

    # --- Behavioral ---
    features['load_factor'] = features['mean'] / features['max'] if features['max'] != 0 else 0
    features['peak_to_avg'] = features['max'] / features['mean'] if features['mean'] != 0 else 0
    features['cv'] = features['std'] / features['mean'] if features['mean'] != 0 else 0

    # --- Dynamics ---
    diff = np.diff(consumption)
    features['mean_abs_diff'] = np.mean(np.abs(diff)) if len(diff) > 0 else 0
    features['std_diff'] = np.std(diff) if len(diff) > 0 else 0

    # --- Temporal ---
    window = window.copy()
    window['hour'] = window['timestamp'].dt.hour

    day = window[(window['hour'] >= 6) & (window['hour'] < 18)]['consumption']
    night = window[(window['hour'] < 6) | (window['hour'] >= 18)]['consumption']

    day_mean = day.mean() if len(day) > 0 else 0
    night_mean = night.mean() if len(night) > 0 else 0

    features['day_mean'] = day_mean
    features['night_mean'] = night_mean
    features['day_night_ratio'] = day_mean / night_mean if night_mean != 0 else 0

    # --- Theft-sensitive ---
    features['zero_pct'] = np.sum(consumption == 0) / len(consumption)
    features['low_pct'] = np.sum(consumption < (0.2 * features['mean'])) / len(consumption)

    # consecutive low streak
    low_mask = consumption < (0.2 * features['mean'])
    max_streak = 0
    current_streak = 0

    for val in low_mask:
        if val:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0

    features['max_low_streak'] = max_streak

    # --- Label ---
    features['theft'] = window['theft'].max()

    return features

In [ ]:
#Apply to ALL data
WINDOW_SIZE = 24

all_features = []

for meter_id, group in df_fixed.groupby('meter_id'):
    
    group = group.sort_values('timestamp').reset_index(drop=True)

    for i in range(0, len(group) - WINDOW_SIZE + 1, WINDOW_SIZE):
        
        window = group.iloc[i:i + WINDOW_SIZE]

        if len(window) == WINDOW_SIZE:
            feats = extract_features(window)
            feats['meter_id'] = meter_id
            feats['window_id'] = i // WINDOW_SIZE

            all_features.append(feats)

In [ ]:
#Create Dataset

In [18]:
features_df = pd.DataFrame(all_features)



In [23]:
features_df

,mean,std,min,max,median,range,load_factor,peak_to_avg,cv,mean_abs_diff,std_diff,day_mean,night_mean,day_night_ratio,zero_pct,low_pct,max_low_streak,theft,meter_id,window_id
0,0.006529,0.007243,0.00035,0.027600,0.002825,0.027250,0.236564,4.227186,1.109299,0.007322,0.010007,0.007329,0.005729,1.279273,0.000000,0.333333,5,0.0,BR02,0
1,0.000913,0.003028,0.00000,0.011170,0.000000,0.011170,0.081729,12.235509,3.317322,0.000486,0.002188,0.000000,0.001826,0.000000,0.916667,0.916667,22,0.0,BR02,1
2,0.004579,0.005430,0.00000,0.020250,0.001650,0.020250,0.226132,4.422202,1.185853,0.003646,0.005730,0.005567,0.003592,1.549884,0.125000,0.375000,6,0.0,BR02,2
3,0.000101,0.000333,0.00000,0.001209,0.000000,0.001209,0.083183,12.021666,3.316631,0.000053,0.000247,0.000000,0.000201,0.000000,0.916667,0.916667,22,0.0,BR02,3
4,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0,0.0,BR02,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7294,0.023281,0.014353,0.00930,0.086550,0.020275,0.077250,0.268992,3.717584,0.616521,0.008098,0.020453,0.023850,0.022712,1.050083,0.000000,0.000000,0,0.0,BR51,299
7295,0.021192,0.004080,0.01440,0.028600,0.020400,0.014200,0.740967,1.349587,0.192550,0.003409,0.004140,0.019537,0.022846,0.855189,0.000000,0.000000,0,0.0,BR51,300
7296,0.020367,0.003648,0.01510,0.032100,0.019925,0.017000,0.634476,1.576105,0.179137,0.002885,0.003789,0.018950,0.021783,0.869931,0.000000,0.000000,0,0.0,BR51,301
7297,0.018675,0.003568,0.01305,0.027650,0.017550,0.014600,0.675407,1.480589,0.191076,0.002857,0.004236,0.019167,0.018183,1.054079,0.000000,0.000000,0,0.0,BR51,302


In [ ]:
# features look correct

# You have all critical groups:

# Statistical ✔
# Behavioral ✔
# Dynamics ✔
# Temporal ✔
# Theft-sensitive ✔
# ✔ Values sanity

# Examples:

# cv > 1 → high variability ✔
# peak_to_avg ~ 12 → spike behavior ✔
# zero_pct, low_pct behaving correctly ✔
# max_low_streak capturing long flat periods ✔

# 👉 This confirms your simulation patterns are being captured

In [24]:
features_df['theft'].value_counts()

theft
0.0    6610
1.0     689
Name: count, dtype: int64

In [25]:
# Optional: reorder columns (clean structure)
cols = [
    'meter_id', 'window_id',
    'mean', 'std', 'min', 'max', 'median', 'range',
    'load_factor', 'peak_to_avg', 'cv',
    'mean_abs_diff', 'std_diff',
    'day_mean', 'night_mean', 'day_night_ratio',
    'zero_pct', 'low_pct', 'max_low_streak',
    'theft'
]

features_df = features_df[cols]

In [27]:
features_df.to_csv("../data/processed/window_features.csv", index=False)
print("Feature dataset saved ✅")

Feature dataset saved ✅
